In [ ]:
import model_library as ml
import learning_rates as lr

import numpy as np

from keras.callbacks import ModelCheckpoint, EarlyStopping

from sklearn.ensemble import RandomForestClassifier

In [ ]:
data = np.load('../data/cm0/ecc_cm0_1024traces_2000pts.npz')

data_full_test = np.load('../data/cm0/cm0_2000.npz')

In [15]:
list(data)

['X_train',
 'Y_train',
 'Y_bits_train',
 'X_test',
 'Y_test',
 'Y_bits_test',
 'X_holdout',
 'Y_holdout',
 'Y_bits_holdout']

In [16]:
X_train,Y_train = data['X_train'],data['Y_bits_train']
X_test,Y_test = data['X_test'],data['Y_bits_test']
X_holdout,Y_holdout = data['X_holdout'],data['Y_bits_holdout']
print(X_train.shape,Y_train.shape)
print(X_test.shape,Y_test.shape)
print(X_holdout.shape,Y_holdout.shape)

X_test_full, Y_test_full = data_full_test['X'], data_full_test['Y']
X_test_full.shape

print(np.mean(np.square(X_test_full[1] - X_test[1])))
print(np.mean(np.square(X_test_full[171] - X_test[171])))
print(Y_test_full[0])
print(Y_test[0])



(1024, 2000) (1024, 256)
(1024, 2000) (1024, 256)
(1024, 2000) (1024, 256)
4.1514652333257746e-17
3.0711365509453064e-17
[1 1 0 0 0 1 1 0 0 0 0 0 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0
 1 1 0 0 1 1 0 1 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 1 0 1 1 0 1 0 1 1 1 0 0 0 1
 0 0 0 0 1 0 1 0 0 1 0 1 0 0 1 1 0 1 0 0 1 0 1 1 0 0 1 1 1 1 0 0 0 0 0 0 1
 0 0 0 1 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 1 0 0 1 1 1 1 1 0 1 1 1 0 0 0 1 0
 0 1 1 1 0 1 0 1 1 0 1 0 0 0 0 0 0 1 1 1 0 0 0 0 1 0 0 1 1 0 0 0 0 0 1 0 0
 1 0 0 0 1 0 1 0 1 1 0 0 0 0 0 0 0 1 0 0 1 1 1 1 0 0 0 1 0 1 1 1 0 1 1 1 0
 0 1 0 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 1 1 1 0 1 1 1 0 0 1 0 1 1 0 0]
[1 1 0 0 0 1 1 0 0 0 0 0 1 1 1 0 1 0 1 0 1 1 0 0 0 0 0 1 0 1 1 0 0 1 0 0 0
 1 1 0 0 1 1 0 1 1 1 1 1 1 1 0 0 1 1 0 0 1 0 0 1 0 1 1 0 1 0 1 1 1 0 0 0 1
 0 0 0 0 1 0 1 0 0 1 0 1 0 0 1 1 0 1 0 0 1 0 1 1 0 0 1 1 1 1 0 0 0 0 0 0 1
 0 0 0 1 0 1 0 0 0 0 0 1 1 0 1 0 1 1 0 1 1 0 0 1 1 1 1 1 0 1 1 1 0 0 0 1 0
 0 1 1 1 0 1 0 1 1 0 1 0 0 0 0 0 0 1 1 1 0 0 0 0 1 0 0 1 1 

In [17]:
rf = RandomForestClassifier(max_depth=10)

In [18]:
rf.fit(X_test_full, Y_test_full[:,163])

RandomForestClassifier(max_depth=10)

In [19]:
score_rf_train = rf.score(X_train,Y_train[:,163])
score_rf_test = rf.score(X_test,Y_test[:,163])
score_rf_holdout = rf.score(X_holdout,Y_holdout[:,163])

print("RF train accuracy: ", score_rf_train)
print("RF test accuracy: ", score_rf_test)
print("RF holdout accuracy: ", score_rf_holdout)


RF train accuracy:  0.9892578125
RF test accuracy:  1.0
RF holdout accuracy:  0.9619140625


In [20]:
nc = ml.net_ches_2018(256, 20, 100, depth=20, output_activation='sigmoid')

In [21]:
nc.compile(optimizer='adam',loss='binary_crossentropy',metrics='binary_accuracy')

In [22]:
check = ModelCheckpoint('nc_2000_fresh.keras', monitor='val_loss', save_best_only=True, save_weights_only=False, mode='auto', period=1)
early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
lr_scheduler = lr.lr_cyclic_sawtooth(epochs=20, lr_max=1e-3, lr_min=1e-6, shift=10)

In [23]:
nc.fit(X_test_full,Y_test_full,batch_size=32,epochs=100,validation_split=0.1, callbacks=[check, early_stop, lr_scheduler])

Epoch 1/100
231/231 [==============================] - 11s 19ms/step - loss: 2.8674 - binary_accuracy: 0.5000 - val_loss: 2.0348 - val_binary_accuracy: 0.4998 - lr: 1.0000e-06
Epoch 2/100
231/231 [==============================] - 4s 17ms/step - loss: 0.8412 - binary_accuracy: 0.5109 - val_loss: 0.9106 - val_binary_accuracy: 0.5001 - lr: 1.0090e-04
Epoch 3/100
231/231 [==============================] - 4s 15ms/step - loss: 0.7336 - binary_accuracy: 0.5570 - val_loss: 0.7917 - val_binary_accuracy: 0.5202 - lr: 2.0080e-04
Epoch 4/100
231/231 [==============================] - 4s 15ms/step - loss: 0.6787 - binary_accuracy: 0.6440 - val_loss: 0.6806 - val_binary_accuracy: 0.6450 - lr: 3.0070e-04
Epoch 5/100
231/231 [==============================] - 3s 15ms/step - loss: 0.5901 - binary_accuracy: 0.7285 - val_loss: 0.6143 - val_binary_accuracy: 0.7111 - lr: 4.0060e-04
Epoch 6/100
231/231 [==============================] - 3s 15ms/step - loss: 0.5028 - binary_accuracy: 0.7899 - val_loss: 0.5

In [ ]:
nc.load_weights('../data/pretrained/nc_2000_fresh.keras')

In [25]:
nc.evaluate(X_train,Y_train,batch_size=32)
nc.evaluate(X_test,Y_test,batch_size=32)
nc.evaluate(X_holdout,Y_holdout,batch_size=32)

pred_holdout_model_fresh = nc.predict(X_holdout)


32/32 [==============================] - 0s 2ms/step


In [ ]:
# now load the pre-trained model that was trained on the full test set for longer (8192 traces)
nc.load_weights('../data/pretrained/nc_2000.keras')

In [27]:
nc.evaluate(X_train,Y_train,batch_size=32)
nc.evaluate(X_test,Y_test,batch_size=32)
nc.evaluate(X_holdout,Y_holdout,batch_size=32)

pred_holdout = nc.predict(X_holdout)

 1/32 [..............................] - ETA: 0s - loss: 0.0322 - binary_accuracy: 1.0000

32/32 [==============================] - 0s 2ms/step


In [28]:
# check for how many traces the pre-trained model predicts the whole key correctly

acc_key = np.mean((pred_holdout > 0.5) == Y_holdout, axis=1)
print(acc_key.shape)

print("Max accuracy: ", np.max(acc_key))
print("Min accuracy: ", np.min(acc_key))
print("Mean accuracy: ", np.mean(acc_key))

print("Percentage of traces with perfect accuracy: ", 100 * np.sum(acc_key == 1.0)/len(acc_key))

(1024,)
Max accuracy:  1.0
Min accuracy:  0.9921875
Mean accuracy:  0.9998588562011719
Percentage of traces with perfect accuracy:  96.58203125


In [29]:
acc_key_model_fresh = np.mean((pred_holdout_model_fresh > 0.5) == Y_holdout, axis=1)
print(acc_key_model_fresh.shape)

print("Max accuracy: ", np.max(acc_key_model_fresh))
print("Min accuracy: ", np.min(acc_key_model_fresh))
print("Mean accuracy: ", np.mean(acc_key_model_fresh))

print("Percentage of traces with perfect accuracy: ", 100 * np.sum(acc_key_model_fresh == 1.0)/len(acc_key_model_fresh))


(1024,)
Max accuracy:  1.0
Min accuracy:  0.9921875
Mean accuracy:  0.9998512268066406
Percentage of traces with perfect accuracy:  96.6796875
